# IndoBERT Sentiment Training (Penginapan)
Notebook ini diformat menjadi versi sangat ringkas (hanya 1 cell teks dan 1 cell *code*).
Memuat seluruh _pipeline_ IndoBERT dari awal hingga akhir (_End-to-End_):
1. Load data
2. Tokenisasi
3. Training
4. Inferensi & Bayesian Shrinkage
5. Gabung (_Merge_) ke _Primary Dataset_


In [ ]:
# Install dependencies (Wajib di Kaggle karena library evaluate belum terinstall bawaan)
!pip install transformers datasets evaluate accelerate scikit-learn numpy pandas torch

import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.utils.class_weight import compute_class_weight
from torch import nn
import torch.nn.functional as F
import evaluate
import os

# Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}\n")

# ==========================================
# KONFIGURASI PATH (WAJIB UBAH JIKA DI KAGGLE)
# ==========================================
# Ganti dengan path dataset LABELLED Anda di Kaggle (contoh: "/kaggle/input/dataset/PENGINAPAN_FASE_A_LABELLED_2026-06-14.csv")
DATA_PATH = "/kaggle/input/datasets/muhammadfauzanlubada/datasestr/PENGINAPAN_FASE_A_LABELLED_2026-06-14.csv"

# Ganti dengan path dataset PRIMARY Anda di Kaggle (contoh: "/kaggle/input/dataset/PENGINAPAN_PRIMARY_DATA_FOCUS_CANDIDATE_SENTIMENT_UPDATED_2026-06-14.csv")
PRIMARY_PATH = "/kaggle/input/datasets/muhammadfauzanlubada/datasestr/PENGINAPAN_PRIMARY_DATA_FOCUS_CANDIDATE_SENTIMENT_UPDATED_2026-06-14.csv"

# 1. LOAD DATASET
print("1. Loading Dataset...")
df = pd.read_csv(DATA_PATH, low_memory=False)

label_map = {"Positif": 0, "Negatif": 1, "Netral": 2}
df['label_num'] = df['sentiment_label'].map(label_map)

# Split data: 80% Train, 20% Test
df_train = df.sample(frac=0.8, random_state=42)
df_test = df.drop(df_train.index)

dataset_train = Dataset.from_pandas(df_train[['cleaned_text', 'label_num']].rename(columns={'label_num': 'label'}))
dataset_test = Dataset.from_pandas(df_test[['cleaned_text', 'label_num']].rename(columns={'label_num': 'label'}))
print(f"Data latih: {len(dataset_train)} baris | Data uji: {len(dataset_test)} baris\n")

# 2. TOKENIZATION
print("2. Tokenisasi IndoBERT...")
MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["cleaned_text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = dataset_train.map(tokenize_function, batched=True)
tokenized_test = dataset_test.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(device)
print("Tokenisasi & Model loading selesai!\n")

# 3. TRAINING
print("3. Setup Training & Class Weights...")
class_weights = compute_class_weight('balanced', classes=np.unique(df_train['label_num']), y=df_train['label_num'])
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Class Weights Tensor: {class_weights_tensor}")

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",  
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Memulai Training...")
trainer.train()
print("Training Selesai!\n")

# 4. INFERENCE & BAYESIAN SHRINKAGE
print("4. Inferensi & Agregasi Bayesian Shrinkage...")
def predict_score(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probas = F.softmax(logits, dim=-1).cpu().numpy()[0]
        
    pos_val, neg_val = probas[0], probas[1]
    score = pos_val - neg_val
    confidence = np.max(probas)
    return score, confidence

scores, confidences = [], []
for idx, text in enumerate(df['cleaned_text']):
    s, c = predict_score(text)
    scores.append(s)
    confidences.append(c)
    if (idx + 1) % 1000 == 0:
        print(f"Prediksi {idx + 1}/{len(df)} selesai.")

df['review_sentiment_score'] = scores
df['review_confidence'] = confidences

K_SHRINKAGE = 50.0
global_mean_score = np.mean(scores)

df_agg = df.groupby('target_penginapan_id').agg(
    hotel_review_count_analyzed=('reviewId', 'count'),
    hotel_sentiment_score=('review_sentiment_score', 'mean'),
    hotel_sentiment_confidence_mean=('review_confidence', 'mean')
).reset_index()

df_agg['hotel_adjusted_sentiment_score'] = (
    (df_agg['hotel_review_count_analyzed'] * df_agg['hotel_sentiment_score'] + K_SHRINKAGE * global_mean_score) 
    / (df_agg['hotel_review_count_analyzed'] + K_SHRINKAGE)
)

def get_sentiment_label(score):
    if score >= 0.6: return "Sangat Positif"
    if score >= 0.2: return "Positif"
    if score >= -0.2: return "Netral"
    if score >= -0.6: return "Negatif"
    return "Sangat Negatif"

df_agg['hotel_adjusted_sentiment_label'] = df_agg['hotel_adjusted_sentiment_score'].apply(get_sentiment_label)
print(f"Agregasi selesai. Total penginapan unik: {len(df_agg)}\n")

# 5. MERGE KE PRIMARY DATASET
print("5. Merge ke Primary Dataset (900 baris)...")
df_primary = pd.read_csv(PRIMARY_PATH, low_memory=False)

# Buang kolom sentimen lama di primary dataset agar tidak duplikat (_x, _y) saat di-merge
cols_to_drop = [c for c in df_agg.columns if c in df_primary.columns and c != 'target_penginapan_id']
df_primary = df_primary.drop(columns=cols_to_drop, errors='ignore')

# Merge menggunakan target_penginapan_id (bukan placeId)
df_final = pd.merge(df_primary, df_agg, on='target_penginapan_id', how='left')
coverage_akhir = df_final['hotel_adjusted_sentiment_score'].notna().sum()

print(f"Dataset Primer Total: {len(df_final)} baris")
print(f"Penginapan yang memiliki skor sentimen AI: {coverage_akhir} properties")

EXPORT_PATH = r"D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\02_Curated\PENGINAPAN_PRIMARY_FINAL_INDOBERT.csv"
# (Di Kaggle, ganti EXPORT_PATH menjadi "/kaggle/working/PENGINAPAN_PRIMARY_FINAL_INDOBERT.csv")
df_final.to_csv(EXPORT_PATH, index=False)
print(f"\nPIPELINE SELESAI. File tersimpan di: {EXPORT_PATH}")

# 6. MEMBUNGKUS FILE OUTPUT & MODEL MENJADI ZIP UNTUK DIDOWNLOAD DI KAGGLE
print("\n6. MEMBUNGKUS OUTPUT MENJADI ZIP...")
import shutil
from IPython.display import display, FileLink

# Kita bungkus direktori saat ini ("./") ke dalam zip
zip_path = "/kaggle/working/penginapan_indobert_final"
OUTPUT_DIR = "./"  
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)

print("✅ BERHASIL DIBUNGKUS!")
print("👉 KLIK LINK DI BAWAH INI UNTUK MENDOWNLOAD SEMUA HASIL (CSV & MODEL) DALAM BENTUK ZIP:")
display(FileLink(r'penginapan_indobert_final.zip'))
